In [1]:
import random
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# CONFIGURATION
# =========================
NUM_SAMPLES_TOTAL = 12000
TRAIN_SIZE = 8000
TEST_SIZE = 2000
FINAL_TEST_SIZE = 2000

# Output paths
TRAIN_PATH = Path("../../../data/training")
TEST_PATH = Path("../../../data/test")
FINAL_TEST_PATH = Path("../../../data/final test")

TRAIN_PATH.mkdir(parents=True, exist_ok=True)
TEST_PATH.mkdir(parents=True, exist_ok=True)
FINAL_TEST_PATH.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = ["Temperature", "Pressure", "RPM", "Vibration"]
WINDOW_LEN = 4

# =========================
# HELPERS (Engine Off logic as in M1)
# =========================
def gen_off_timestep():
    """
    One OFF timestep using same value ranges as EngineOff (M1).
    - Randomly picks cold (0) or cooling (1).
    - Temperature: cold [-10, 35], cooling [60, 120]
    - Pressure: uniform [0.92, 1.02]
    - RPM: 0.0
    - Vibration: normal low; if temp >= 95 (cooling), slightly higher
    """
    state = random.choice([0, 1])  # 0=cold, 1=cooling

    if state == 0:
        temperature = random.uniform(-10, 35)
        vibration = random.uniform(0.0001, 0.001)
    else:
        temperature = random.uniform(60, 120)
        if temperature >= 95:
            vibration = random.uniform(0.002, 0.005)
        else:
            vibration = random.uniform(0.0001, 0.001)

    pressure = random.uniform(0.92, 1.02)
    rpm = 0.0
    return [temperature, pressure, rpm, vibration]

# =========================
# GENERATE ONE ENGINE START WINDOW (4 timesteps)
# =========================
def gen_engine_start_window():
    """
    Build a 4-step window:
      - n_off in {1,2,3} OFF timesteps first (same logic as EngineOff).
      - Remaining steps are ON with:
          * Temp += 3..7 each ON step
          * RPM first ON: 1500..3000
          * RPM second ON: previous - (400..600)
          * RPM third ON (if exists): previous - (100..300)
          * Pressure on first ON: 0.7..0.9 (drop), then normal 0.92..1.02
          * Vibration on first ON: 0.3..0.6 (spike), then 0.05..0.15 (settle)
    Label: "Engine Start"
    """
    n_off = random.choice([1, 2, 3])
    n_on = WINDOW_LEN - n_off

    window = []

    # OFF part
    last_temp = None
    for _ in range(n_off):
        t = gen_off_timestep()
        window.append(t)
        last_temp = t[0]  # keep last temperature to start ON ramp

    # ON part
    if n_on > 0:
        # First ON step
        temp = last_temp + random.uniform(3, 7)
        rpm = random.uniform(1500, 3000)
        pressure = random.uniform(0.7, 0.9)       # pressure drop
        vibration = random.uniform(0.3, 0.6)      # vibration spike
        window.append([temp, pressure, rpm, vibration])

        # Second ON step (if exists): drop 400..600
        if n_on >= 2:
            temp = temp + random.uniform(3, 7)
            rpm = max(0.0, rpm - random.uniform(400, 600))
            pressure = random.uniform(0.9, 0.95)  # back to normal range
            vibration = random.uniform(0.15, 0.3) # settle
            window.append([temp, pressure, rpm, vibration])

        # Third ON step (if exists): drop 100..300
        if n_on == 3:
            temp = temp + random.uniform(3, 7)
            rpm = max(0.0, rpm - random.uniform(100, 300))
            pressure = random.uniform(0.98, 1.02)  # back to normal range
            vibration = random.uniform(0.05, 0.15) # settle
            window.append([temp, pressure, rpm, vibration])

    assert len(window) == WINDOW_LEN
    return window, "Engine Start"

# =========================
# GENERATE DATASET
# =========================

def generate_dataset(num_samples):
    X_list = []
    y_list = []
    rows = []

    for seq_num in range(num_samples):
        window, label = gen_engine_start_window()
        X_list.append(window)
        y_list.append(label)

        # For CSV: long format with Time, Sequence, variables, State
        for t_idx, (temp, pres, rpm, vib) in enumerate(window, start=1):
            rows.append({
                "Time": t_idx,               # 1..WINDOW_LEN
                "Sequence": seq_num + 1,     # 1..num_samples
                "Temperature": temp,
                "Pressure": pres,
                "RPM": rpm,
                "Vibration": vib,
                "State": label               # keep label as-is: "Engine Start"
            })

    X = np.array(X_list, dtype=np.float32)  # (N, 4, 4)
    y = np.array(y_list)                    # (N,)
    df = pd.DataFrame(rows, columns=["Time","Sequence","Temperature","Pressure","RPM","Vibration","State"])
    return X, y, df

# =========================
# BUILD SPLITS & SAVE
# =========================
# Training
X_train, y_train, df_train = generate_dataset(TRAIN_SIZE)
np.save(TRAIN_PATH / "EngineStart_training_X.npy", X_train)
np.save(TRAIN_PATH / "EngineStart_training_y.npy", y_train)
df_train.to_csv(TRAIN_PATH / "EngineStart_training.csv", index=False)

# Test
X_test, y_test, df_test = generate_dataset(TEST_SIZE)
np.save(TEST_PATH / "EngineStart_test_X.npy", X_test)
np.save(TEST_PATH / "EngineStart_test_y.npy", y_test)
df_test.to_csv(TEST_PATH / "EngineStart_test.csv", index=False)

# Final test
X_final, y_final, df_final = generate_dataset(FINAL_TEST_SIZE)
np.save(FINAL_TEST_PATH / "EngineStart_final test_X.npy", X_final)
np.save(FINAL_TEST_PATH / "EngineStart_final test_y.npy", y_final)
df_final.to_csv(FINAL_TEST_PATH / "EngineStart_final test.csv", index=False)

print("✅ Engine Start dataset created.")
print("Train X:", X_train.shape, "| Test X:", X_test.shape, "| Final X:", X_final.shape)


✅ Engine Start dataset created.
Train X: (8000, 4, 4) | Test X: (2000, 4, 4) | Final X: (2000, 4, 4)
